In [1]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os

# Load credentials
user = "root"  # or your MySQL username
password = quote_plus(os.getenv("SQL_PASSWORD"))  # handles special characters
host = "localhost"
database = "sakila"

# Create connection string
connection_string = f"mysql+pymysql://{user}:{password}@{host}/{database}"

# Create engine
engine = create_engine(connection_string)

# Test connection
df = pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)
print(df.head())

   customer_id  store_id first_name last_name  \
0            1         1       MARY     SMITH   
1            2         1   PATRICIA   JOHNSON   
2            3         1      LINDA  WILLIAMS   
3            4         2    BARBARA     JONES   
4            5         1  ELIZABETH     BROWN   

                                 email  address_id  active  \
0        MARY.SMITH@sakilacustomer.org           5       1   
1  PATRICIA.JOHNSON@sakilacustomer.org           6       1   
2    LINDA.WILLIAMS@sakilacustomer.org           7       1   
3     BARBARA.JONES@sakilacustomer.org           8       1   
4   ELIZABETH.BROWN@sakilacustomer.org           9       1   

          create_date         last_update  
0 2006-02-14 22:04:36 2006-02-15 04:57:20  
1 2006-02-14 22:04:36 2006-02-15 04:57:20  
2 2006-02-14 22:04:36 2006-02-15 04:57:20  
3 2006-02-14 22:04:36 2006-02-15 04:57:20  
4 2006-02-14 22:04:36 2006-02-15 04:57:20  


In [ ]:
# Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) 
# from the Sakila database as a Pandas DataFrame. The function should take in three parameters:
# engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
# month: an integer representing the month for which rental data is to be retrieved.
# year: an integer representing the year for which rental data is to be retrieved.
# The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, 
# and return it as a pandas DataFrame.

def rentals_month(engine, month, year):
    query = """
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = %s
      AND YEAR(rental_date) = %s;
    """
    
    df = pd.read_sql(query, engine, params=(month, year))
    return df

In [ ]:
#Check the usage
df_may_2005 = rentals_month(engine, 5, 2005)
print(df_may_2005.head())

   rental_id         rental_date  inventory_id  customer_id  \
0          1 2005-05-24 22:53:30           367          130   
1          2 2005-05-24 22:54:33          1525          459   
2          3 2005-05-24 23:03:39          1711          408   
3          4 2005-05-24 23:04:41          2452          333   
4          5 2005-05-24 23:05:21          2079          222   

          return_date  staff_id         last_update  
0 2005-05-26 22:04:30         1 2006-02-15 21:30:53  
1 2005-05-28 19:40:33         1 2006-02-15 21:30:53  
2 2005-06-01 22:12:39         1 2006-02-15 21:30:53  
3 2005-06-03 01:43:41         2 2006-02-15 21:30:53  
4 2005-06-02 04:33:21         1 2006-02-15 21:30:53  


In [ ]:
# Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month 
# and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.
# The function should also include the month and year as parameters and use them to name the new column according to the month and year, 
# for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".
# Hint: Consider making use of pandas groupby()

In [17]:
def rental_count_month(df, month, year):
    # Create dynamic column name
    col_name = f"rentals_{month:02d}_{year}"
    
    # Group by customer_id and count rentals
    result = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=col_name)
    )
    
    return result

In [ ]:
#Check the example of the output
df_may_2005 = rentals_month(engine, 5, 2005)

df_counts = rental_count_month(df_may_2005, 5, 2005)

print(df_counts.head())

   customer_id  rentals_05_2005
0            1                2
1            2                1
2            3                2
3            5                3
4            6                3


In [ ]:
#Create a Python function called compare_rentals that takes two DataFrames as input containing the number 
# of rentals made by each customer in different months and years. 
#The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [19]:
def compare_rentals(df1, df2):
    # Merge both DataFrames on customer_id
    merged = pd.merge(df1, df2, on="customer_id", how="outer").fillna(0)
    
    # Identify rental columns (exclude customer_id)
    rental_cols = [col for col in merged.columns if col != "customer_id"]
    
    # Compute difference (second month - first month)
    merged["difference"] = merged[rental_cols[1]] - merged[rental_cols[0]]
    
    return merged

In [22]:
# Example of usage:
# Get raw rental data
df_may_2005 = rentals_month(engine, 5, 2005)
df_june_2005 = rentals_month(engine, 6, 2005)

# Aggregate per customer
df_may = rental_count_month(df_may_2005, 5, 2005)
df_june = rental_count_month(df_june_2005, 6, 2005)

# Compare
comparison = compare_rentals(df_may, df_june)

print(comparison.head())

   customer_id  rentals_05_2005  rentals_06_2005  difference
0            1              2.0              7.0         5.0
1            2              1.0              1.0         0.0
2            3              2.0              4.0         2.0
3            4              0.0              6.0         6.0
4            5              3.0              5.0         2.0
